# Advanced Practice Problems: Python Custom Classes — Part 2

This notebook **extends** the earlier notebook on custom classes.

It does **not** repeat the earlier exercises on basic `Rectangle`, `Student`, `BankAccount`, or `Product` classes. Instead, it builds on that foundation and introduces more advanced design patterns and Pythonic features for custom classes.

Topics covered here:
- class methods and alternate constructors
- static methods
- inheritance and method overriding
- `super()`
- composition
- custom iterables and `__len__`
- indexing and slicing with `__getitem__`
- containment with `__contains__`
- truthiness with `__bool__`
- hashing with `__hash__`
- callable objects with state
- rich comparisons with `functools.total_ordering`
- data validation and invariants
- lightweight object protocols

---
## Problem 1 — Alternate Constructors with `@classmethod`

Create a `Book` class with:
- `title`
- `author`
- `pages`

Requirements:
1. Validate that `pages` is a positive integer.
2. Implement `__repr__`.
3. Add a class method `from_string()` that builds a `Book` from a string in this format:
   `'The Hobbit|J.R.R. Tolkien|310'`
4. Add another class method `short_story()` that creates a short book with 50 pages.

In [1]:
class Book:
    def __init__(self, title, author, pages):
        if not isinstance(pages, int) or pages <= 0:
            raise ValueError('pages must be a positive integer')
        self.title = title
        self.author = author
        self.pages = pages

    def __repr__(self):
        return f"Book(title={self.title!r}, author={self.author!r}, pages={self.pages})"

    @classmethod
    def from_string(cls, data):
        title, author, pages = data.split('|')
        return cls(title, author, int(pages))

    @classmethod
    def short_story(cls, title, author):
        return cls(title, author, 50)


b1 = Book.from_string('The Hobbit|J.R.R. Tolkien|310')
b2 = Book.short_story('The Tell-Tale Heart', 'Edgar Allan Poe')
print(b1)
print(b2)

Book(title='The Hobbit', author='J.R.R. Tolkien', pages=310)
Book(title='The Tell-Tale Heart', author='Edgar Allan Poe', pages=50)


---
## Problem 2 — Use `@staticmethod` for Validation Helpers

Create an `EmailAddress` class.

Requirements:
1. Store a validated email string.
2. Use a static method `is_valid_email()`.
3. Reject invalid emails in `__init__`.
4. Add `domain` as a read-only property.

In [2]:
class EmailAddress:
    def __init__(self, email):
        if not self.is_valid_email(email):
            raise ValueError('invalid email address')
        self._email = email

    @staticmethod
    def is_valid_email(email):
        return isinstance(email, str) and '@' in email and '.' in email.split('@')[-1]

    @property
    def email(self):
        return self._email

    @property
    def domain(self):
        return self._email.split('@')[1]

    def __repr__(self):
        return f'EmailAddress({self.email!r})'


e = EmailAddress('alice@example.com')
print(e)
print(e.domain)
print(EmailAddress.is_valid_email('bad-email'))

EmailAddress('alice@example.com')
example.com
False


---
## Problem 3 — Inheritance Basics: Employee and Manager

Create:
- `Employee(name, salary)`
- `Manager(name, salary, department)` that inherits from `Employee`

Requirements:
1. Validate that salary is non-negative.
2. Give both classes a readable `__repr__`.
3. Add a `give_raise(percent)` method in `Employee`.
4. Override `__repr__` in `Manager`.
5. Use `super()` correctly.

In [3]:
class Employee:
    def __init__(self, name, salary):
        if salary < 0:
            raise ValueError('salary cannot be negative')
        self.name = name
        self.salary = salary

    def give_raise(self, percent):
        if percent < 0:
            raise ValueError('raise percent cannot be negative')
        self.salary *= (1 + percent / 100)

    def __repr__(self):
        return f'Employee(name={self.name!r}, salary={self.salary:.2f})'


class Manager(Employee):
    def __init__(self, name, salary, department):
        super().__init__(name, salary)
        self.department = department

    def __repr__(self):
        return (
            f'Manager(name={self.name!r}, salary={self.salary:.2f}, '
            f'department={self.department!r})'
        )


emp = Employee('Lina', 50000)
mgr = Manager('David', 80000, 'Engineering')
emp.give_raise(10)
mgr.give_raise(5)
print(emp)
print(mgr)

Employee(name='Lina', salary=55000.00)
Manager(name='David', salary=84000.00, department='Engineering')


---
## Problem 4 — Method Overriding with Polymorphism

Create a base class `Shape` with an `area()` method that raises `NotImplementedError`.

Then create:
- `Circle(radius)`
- `Square(side)`

Requirements:
1. Override `area()` in each subclass.
2. Put several shapes into a list.
3. Compute the total area using polymorphism.

In [4]:
import math


class Shape:
    def area(self):
        raise NotImplementedError('subclasses must implement area()')


class Circle(Shape):
    def __init__(self, radius):
        if radius <= 0:
            raise ValueError('radius must be positive')
        self.radius = radius

    def area(self):
        return math.pi * self.radius ** 2

    def __repr__(self):
        return f'Circle(radius={self.radius})'


class Square(Shape):
    def __init__(self, side):
        if side <= 0:
            raise ValueError('side must be positive')
        self.side = side

    def area(self):
        return self.side ** 2

    def __repr__(self):
        return f'Square(side={self.side})'


shapes = [Circle(2), Square(3), Circle(1.5), Square(10)]
total_area = sum(shape.area() for shape in shapes)
print(shapes)
print(f'Total area: {total_area:.2f}')

[Circle(radius=2), Square(side=3), Circle(radius=1.5), Square(side=10)]
Total area: 128.63


---
## Problem 5 — Composition: A Library Owns Books

Create a `Library` class that uses composition.

Requirements:
1. A library has a name and a collection of `Book` objects.
2. Add `add_book(book)`.
3. Add `total_pages()`.
4. Make the library printable.
5. Reject non-`Book` objects.

In [5]:
class Book:
    def __init__(self, title, author, pages):
        if not isinstance(pages, int) or pages <= 0:
            raise ValueError('pages must be a positive integer')
        self.title = title
        self.author = author
        self.pages = pages

    def __repr__(self):
        return f'Book({self.title!r}, {self.author!r}, {self.pages})'


class Library:
    def __init__(self, name):
        self.name = name
        self._books = []

    def add_book(self, book):
        if not isinstance(book, Book):
            raise TypeError('book must be a Book instance')
        self._books.append(book)

    def total_pages(self):
        return sum(book.pages for book in self._books)

    def __repr__(self):
        return f'Library(name={self.name!r}, books={len(self._books)})'


library = Library('City Library')
library.add_book(Book('Dune', 'Frank Herbert', 412))
library.add_book(Book('1984', 'George Orwell', 328))
library.add_book(Book('Hamlet', 'William Shakespeare', 160))
print(library)
print(library.total_pages())

Library(name='City Library', books=3)
900


---
## Problem 6 — Make a Class Iterable

Extend the `Library` idea and make a class iterable.

Create a `Playlist` class that stores songs.

Requirements:
1. Add songs with `add_song(song)`.
2. Implement `__iter__`.
3. Implement `__len__`.
4. Show that it works with `for` loops and `len()`.

In [6]:
class Playlist:
    def __init__(self, name):
        self.name = name
        self._songs = []

    def add_song(self, song):
        if not isinstance(song, str) or not song.strip():
            raise ValueError('song must be a non-empty string')
        self._songs.append(song)

    def __iter__(self):
        return iter(self._songs)

    def __len__(self):
        return len(self._songs)

    def __repr__(self):
        return f'Playlist({self.name!r}, songs={len(self)})'


playlist = Playlist('Morning Mix')
playlist.add_song('Here Comes the Sun')
playlist.add_song('Dreams')
playlist.add_song('Viva La Vida')

print(playlist)
print(len(playlist))
for song in playlist:
    print(song)

Playlist('Morning Mix', songs=3)
3
Here Comes the Sun
Dreams
Viva La Vida


---
## Problem 7 — Support Indexing and Slicing

Create a `NumberSequence` class that wraps a list of numbers.

Requirements:
1. Support indexing with `obj[index]`.
2. Support slicing with `obj[start:stop]`.
3. Return a plain list for slices.
4. Validate that all inputs are numeric.

In [7]:
class NumberSequence:
    def __init__(self, values):
        values = list(values)
        if not all(isinstance(v, (int, float)) for v in values):
            raise TypeError('all values must be numeric')
        self._values = values

    def __getitem__(self, index):
        return self._values[index]

    def __repr__(self):
        return f'NumberSequence({self._values})'


seq = NumberSequence([10, 20, 30, 40, 50, 60])
print(seq[0])
print(seq[3])
print(seq[1:5])
print(seq[::-1])

10
40
[20, 30, 40, 50]
[60, 50, 40, 30, 20, 10]


---
## Problem 8 — Implement `__contains__` for Fast Membership Semantics

Create a `TagCloud` class.

Requirements:
1. Store unique tags.
2. Normalize tags to lowercase.
3. Support `in` with `__contains__`.
4. Keep insertion order for display.

In [8]:
class TagCloud:
    def __init__(self):
        self._tags = []
        self._tag_set = set()

    def add(self, tag):
        normalized = tag.strip().lower()
        if not normalized:
            raise ValueError('tag cannot be empty')
        if normalized not in self._tag_set:
            self._tags.append(normalized)
            self._tag_set.add(normalized)

    def __contains__(self, tag):
        return tag.strip().lower() in self._tag_set

    def __iter__(self):
        return iter(self._tags)

    def __repr__(self):
        return f'TagCloud({self._tags})'


cloud = TagCloud()
cloud.add('Python')
cloud.add('Classes')
cloud.add('python')
cloud.add('OOP')
print(cloud)
print('python' in cloud)
print('decorators' in cloud)

TagCloud(['python', 'classes', 'oop'])
True
False


---
## Problem 9 — Control Truthiness with `__bool__`

Create a `Cart` class.

Requirements:
1. Store product names.
2. `bool(cart)` should be `False` when empty and `True` otherwise.
3. Implement `add_item()`.
4. Implement `__len__`.

In [9]:
class Cart:
    def __init__(self):
        self._items = []

    def add_item(self, item):
        if not isinstance(item, str) or not item.strip():
            raise ValueError('item must be a non-empty string')
        self._items.append(item)

    def __len__(self):
        return len(self._items)

    def __bool__(self):
        return len(self) > 0

    def __repr__(self):
        return f'Cart(items={self._items})'


cart = Cart()
print(bool(cart))
cart.add_item('Keyboard')
print(bool(cart))
print(len(cart))

False
True
1


---
## Problem 10 — Hashable Objects for Use in Sets and Dictionaries

Create a `Point2D` class.

Requirements:
1. Two points are equal if their coordinates match.
2. Make points hashable.
3. Show that duplicate points collapse inside a set.
4. Use a point as a dictionary key.

In [10]:
class Point2D:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __eq__(self, other):
        if not isinstance(other, Point2D):
            return NotImplemented
        return (self.x, self.y) == (other.x, other.y)

    def __hash__(self):
        return hash((self.x, self.y))

    def __repr__(self):
        return f'Point2D({self.x}, {self.y})'


p1 = Point2D(1, 2)
p2 = Point2D(1, 2)
p3 = Point2D(3, 4)

points = {p1, p2, p3}
labels = {p1: 'start', p3: 'end'}

print(points)
print(labels[p2])

{Point2D(1, 2), Point2D(3, 4)}
start


---
## Problem 11 — Rich Ordering with `functools.total_ordering`

Create a `Version` class representing software versions like `1.4.2`.

Requirements:
1. Store `major`, `minor`, `patch`.
2. Implement `__eq__` and `__lt__` only.
3. Use `@total_ordering` so the other comparisons work automatically.
4. Show sorting of versions.

In [11]:
from functools import total_ordering


@total_ordering
class Version:
    def __init__(self, major, minor, patch):
        self.major = major
        self.minor = minor
        self.patch = patch

    def _as_tuple(self):
        return (self.major, self.minor, self.patch)

    def __eq__(self, other):
        if not isinstance(other, Version):
            return NotImplemented
        return self._as_tuple() == other._as_tuple()

    def __lt__(self, other):
        if not isinstance(other, Version):
            return NotImplemented
        return self._as_tuple() < other._as_tuple()

    def __repr__(self):
        return f'{self.major}.{self.minor}.{self.patch}'


versions = [
    Version(1, 4, 2),
    Version(1, 10, 0),
    Version(1, 4, 10),
    Version(2, 0, 0),
    Version(1, 4, 1)
]

print(sorted(versions))
print(Version(1, 4, 2) <= Version(1, 4, 10))
print(Version(2, 0, 0) > Version(1, 10, 0))

[1.4.1, 1.4.2, 1.4.10, 1.10.0, 2.0.0]
True
True


---
## Problem 12 — A Callable Object with Internal State

Create a class `RunningAverage`.

Requirements:
1. The object should be callable.
2. Each call adds one number.
3. Each call returns the updated running average.
4. Track how many values have been seen.

In [12]:
class RunningAverage:
    def __init__(self):
        self.total = 0
        self.count = 0

    def __call__(self, value):
        if not isinstance(value, (int, float)):
            raise TypeError('value must be numeric')
        self.total += value
        self.count += 1
        return self.total / self.count

    def __repr__(self):
        return f'RunningAverage(count={self.count}, total={self.total})'


avg = RunningAverage()
print(avg(10))
print(avg(20))
print(avg(40))
print(avg)

10.0
15.0
23.333333333333332
RunningAverage(count=3, total=70)


---
## Problem 13 — A Bounded Counter with Invariants

Create a `BoundedCounter` class.

Requirements:
1. It has `minimum`, `maximum`, and current `value`.
2. `increment()` raises an error if value would exceed maximum.
3. `decrement()` raises an error if value would go below minimum.
4. Expose `value` as a property.
5. Keep the object always valid.

In [13]:
class BoundedCounter:
    def __init__(self, minimum, maximum, value=None):
        if minimum > maximum:
            raise ValueError('minimum cannot be greater than maximum')
        self.minimum = minimum
        self.maximum = maximum
        self._value = minimum if value is None else value
        if not (self.minimum <= self._value <= self.maximum):
            raise ValueError('initial value out of bounds')

    @property
    def value(self):
        return self._value

    def increment(self):
        if self._value >= self.maximum:
            raise OverflowError('counter is already at maximum')
        self._value += 1

    def decrement(self):
        if self._value <= self.minimum:
            raise OverflowError('counter is already at minimum')
        self._value -= 1

    def __repr__(self):
        return (
            f'BoundedCounter(minimum={self.minimum}, maximum={self.maximum}, '
            f'value={self.value})'
        )


counter = BoundedCounter(0, 3)
print(counter)
counter.increment()
counter.increment()
print(counter)
counter.decrement()
print(counter)

BoundedCounter(minimum=0, maximum=3, value=0)
BoundedCounter(minimum=0, maximum=3, value=2)
BoundedCounter(minimum=0, maximum=3, value=1)


---
## Problem 14 — Lightweight Matrix Class

Create a `Matrix2x2` class.

Requirements:
1. Store 4 numbers: `a`, `b`, `c`, `d`.
2. Implement matrix addition with `+`.
3. Implement scalar multiplication with `*`.
4. Implement determinant as a method.
5. Add readable `__repr__`.

In [14]:
class Matrix2x2:
    def __init__(self, a, b, c, d):
        self.a = a
        self.b = b
        self.c = c
        self.d = d

    def __add__(self, other):
        if not isinstance(other, Matrix2x2):
            return NotImplemented
        return Matrix2x2(
            self.a + other.a,
            self.b + other.b,
            self.c + other.c,
            self.d + other.d
        )

    def __mul__(self, scalar):
        if not isinstance(scalar, (int, float)):
            return NotImplemented
        return Matrix2x2(
            self.a * scalar,
            self.b * scalar,
            self.c * scalar,
            self.d * scalar
        )

    def determinant(self):
        return self.a * self.d - self.b * self.c

    def __repr__(self):
        return f'Matrix2x2([{self.a}, {self.b}], [{self.c}, {self.d}])'


m1 = Matrix2x2(1, 2, 3, 4)
m2 = Matrix2x2(10, 20, 30, 40)
print(m1 + m2)
print(m1 * 3)
print(m1.determinant())

Matrix2x2([11, 22], [33, 44])
Matrix2x2([3, 6], [9, 12])
-2


---
## Problem 15 — A Custom Stack Class

Create a stack implementation from scratch.

Requirements:
1. Implement `push`, `pop`, `peek`.
2. Implement `__len__`.
3. Implement `__bool__`.
4. Raise an informative exception when popping from an empty stack.
5. Add `__repr__`.

In [15]:
class Stack:
    def __init__(self):
        self._items = []

    def push(self, item):
        self._items.append(item)

    def pop(self):
        if not self._items:
            raise IndexError('cannot pop from an empty stack')
        return self._items.pop()

    def peek(self):
        if not self._items:
            raise IndexError('cannot peek into an empty stack')
        return self._items[-1]

    def __len__(self):
        return len(self._items)

    def __bool__(self):
        return len(self) > 0

    def __repr__(self):
        return f'Stack({self._items})'


stack = Stack()
stack.push(10)
stack.push(20)
stack.push(30)
print(stack)
print(stack.peek())
print(stack.pop())
print(stack)
print(bool(stack))

Stack([10, 20, 30])
30
30
Stack([10, 20])
True


---
## Problem 16 — Log Entries with Ordering, Hashing, and Filtering

Create a `LogEntry` class.

Requirements:
1. Store `level`, `message`, and numeric `severity`.
2. Two entries are equal if all fields match.
3. Make entries hashable.
4. Make entries sortable by severity.
5. Filter a list of entries to keep only severity >= 40.

In [16]:
from functools import total_ordering


@total_ordering
class LogEntry:
    def __init__(self, level, message, severity):
        self.level = level
        self.message = message
        self.severity = severity

    def __eq__(self, other):
        if not isinstance(other, LogEntry):
            return NotImplemented
        return (self.level, self.message, self.severity) == (
            other.level, other.message, other.severity
        )

    def __lt__(self, other):
        if not isinstance(other, LogEntry):
            return NotImplemented
        return self.severity < other.severity

    def __hash__(self):
        return hash((self.level, self.message, self.severity))

    def __repr__(self):
        return (
            f'LogEntry(level={self.level!r}, message={self.message!r}, '
            f'severity={self.severity})'
        )


entries = [
    LogEntry('INFO', 'Started app', 20),
    LogEntry('ERROR', 'Database unavailable', 50),
    LogEntry('WARNING', 'Low disk space', 40),
    LogEntry('ERROR', 'Database unavailable', 50)
]

print(sorted(entries))
print(set(entries))
serious = [entry for entry in entries if entry.severity >= 40]
print(serious)

[LogEntry(level='INFO', message='Started app', severity=20), LogEntry(level='WARNING', message='Low disk space', severity=40), LogEntry(level='ERROR', message='Database unavailable', severity=50), LogEntry(level='ERROR', message='Database unavailable', severity=50)]
{LogEntry(level='WARNING', message='Low disk space', severity=40), LogEntry(level='ERROR', message='Database unavailable', severity=50), LogEntry(level='INFO', message='Started app', severity=20)}
[LogEntry(level='ERROR', message='Database unavailable', severity=50), LogEntry(level='WARNING', message='Low disk space', severity=40), LogEntry(level='ERROR', message='Database unavailable', severity=50)]


---
## Problem 17 — A GradeBook with Multiple Object Types

Design a small object model using composition.

Create:
- `Assignment(name, max_score)`
- `Submission(assignment, score)`
- `GradeBook(student_name)`

Requirements:
1. `Submission` must reference an `Assignment` object.
2. Score must be between 0 and `max_score`.
3. `GradeBook` stores submissions.
4. `GradeBook.average_percent()` returns the student's average percentage.
5. Make the final output readable.

In [17]:
class Assignment:
    def __init__(self, name, max_score):
        if max_score <= 0:
            raise ValueError('max_score must be positive')
        self.name = name
        self.max_score = max_score

    def __repr__(self):
        return f'Assignment({self.name!r}, max_score={self.max_score})'


class Submission:
    def __init__(self, assignment, score):
        if not isinstance(assignment, Assignment):
            raise TypeError('assignment must be an Assignment object')
        if not (0 <= score <= assignment.max_score):
            raise ValueError('score must be between 0 and max_score')
        self.assignment = assignment
        self.score = score

    def __repr__(self):
        return f'Submission({self.assignment.name!r}, score={self.score})'


class GradeBook:
    def __init__(self, student_name):
        self.student_name = student_name
        self._submissions = []

    def add_submission(self, submission):
        if not isinstance(submission, Submission):
            raise TypeError('submission must be a Submission object')
        self._submissions.append(submission)

    def average_percent(self):
        if not self._submissions:
            return 0.0
        earned = sum(s.score for s in self._submissions)
        possible = sum(s.assignment.max_score for s in self._submissions)
        return (earned / possible) * 100

    def __repr__(self):
        return (
            f'GradeBook(student_name={self.student_name!r}, '
            f'submissions={len(self._submissions)})'
        )


a1 = Assignment('Quiz 1', 20)
a2 = Assignment('Project', 100)
a3 = Assignment('Final Exam', 80)

gb = GradeBook('Nora')
gb.add_submission(Submission(a1, 18))
gb.add_submission(Submission(a2, 91))
gb.add_submission(Submission(a3, 72))

print(gb)
print(f'Average: {gb.average_percent():.2f}%')

GradeBook(student_name='Nora', submissions=3)
Average: 90.50%


---
## Problem 18 — Mini Challenge: Build a Reusable Range Class

Create a custom `MyRange` class similar to Python's built-in `range`.

Requirements:
1. Accept `start`, `stop`, `step`.
2. Support iteration.
3. Support `len()`.
4. Support membership tests with `in`.
5. Support indexing.

This is a good exercise in combining multiple special methods in one class.

In [18]:
class MyRange:
    def __init__(self, start, stop=None, step=1):
        if stop is None:
            start, stop = 0, start
        if step == 0:
            raise ValueError('step cannot be zero')
        self.start = start
        self.stop = stop
        self.step = step

    def _values(self):
        return list(range(self.start, self.stop, self.step))

    def __iter__(self):
        return iter(self._values())

    def __len__(self):
        return len(self._values())

    def __contains__(self, item):
        return item in self._values()

    def __getitem__(self, index):
        return self._values()[index]

    def __repr__(self):
        return f'MyRange(start={self.start}, stop={self.stop}, step={self.step})'


r = MyRange(2, 15, 3)
print(r)
print(list(r))
print(len(r))
print(8 in r)
print(11 in r)
print(r[2])

MyRange(start=2, stop=15, step=3)
[2, 5, 8, 11, 14]
5
True
True
8


---
# Best Practices Reinforced in This Notebook

Key ideas illustrated by these solutions:

1. **Validate early**
   - Reject bad data inside `__init__` or property setters.

2. **Use the right method type**
   - instance methods for object behavior
   - class methods for alternate constructors
   - static methods for utility logic related to the class

3. **Return `NotImplemented` in rich comparisons/operators when appropriate**
   - This lets Python try reflected operations or produce the correct error.

4. **Prefer composition when one object owns or manages other objects**
   - Example: `Library` owns `Book` objects.

5. **Use inheritance only when there is a clear is-a relationship**
   - Example: `Manager` is an `Employee`.

6. **Implement Python protocols for natural behavior**
   - `__iter__`, `__len__`, `__contains__`, `__bool__`, `__getitem__`, `__call__`

7. **Keep representations useful**
   - `__repr__` should help with debugging.

8. **Preserve class invariants**
   - A well-designed class stays valid after every operation.